In [29]:
import pandas as pd
import numpy as np

# 1. Load the original 2024 Kaggle dataset
df_2024 = pd.read_csv('smart_logistics_dataset.csv')
df_2024['Timestamp'] = pd.to_datetime(df_2024['Timestamp'])



In [30]:
# 2. Clone the data to create a baseline for 2025
df_2025 = df_2024.copy()


In [31]:
# Shift the timestamps forward by exactly 365 days
df_2025['Timestamp'] = df_2025['Timestamp'] + pd.DateOffset(days=365)


In [32]:
# Add slight random variations to metrics so it's not a perfect duplicate
df_2025['Waiting_Time'] = df_2025['Waiting_Time'] * np.random.uniform(0.9, 1.1)



In [33]:
# 3. Clone and create data for Jan - May 2026
df_2026 = df_2024[df_2024['Timestamp'].dt.month <= 5].copy()
df_2026['Timestamp'] = df_2026['Timestamp'] + pd.DateOffset(days=730)



In [34]:
# 4. Combine them into a single 2-year production master file
df_master = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)
df_master.to_csv('fleet_telemetry_2024_2026.csv', index=False)

In [35]:
print("Project Start Date:", df_master['Timestamp'].min())
print("Project End Date:", df_master['Timestamp'].max())

Project Start Date: 2024-01-01 11:37:57
Project End Date: 2026-05-31 23:20:31


In [36]:
# Extract the year and count how many entries exist for each
df_master['Timestamp'].dt.year.value_counts().sort_index()

Timestamp
2024    1003
2025    1000
2026     420
Name: count, dtype: int64

In [37]:
# Look at a sample row from 2024, 2025, and 2026 side by side
display(df_master[df_master['Timestamp'].dt.year == 2024].head(2))
display(df_master[df_master['Timestamp'].dt.year == 2025].head(2))
display(df_master[df_master['Timestamp'].dt.year == 2026].head(2))

,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
0,2024-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,38.0,320,4,NaN,60.1,285,1
1,2024-10-30 07:53:51,Truck_6,22.2748,-131.7086,491,In Transit,22.5,54.3,Heavy,16.0,439,7,Weather,80.9,174,1


,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
1000,2025-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,37.170304,320,4,NaN,60.1,285,1
1001,2025-10-30 07:53:51,Truck_6,22.2748,-131.7086,491,In Transit,22.5,54.3,Heavy,15.650654,439,7,Weather,80.9,174,1


,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
2000,2026-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,38.0,320,4,NaN,60.1,285,1
2001,2026-02-04 08:38:56,Truck_4,27.9307,147.5317,480,Delayed,20.7,75.4,Clear,32.0,263,3,NaN,73.3,198,1


In [38]:
pip install mysql-connector-python sqlalchemy


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
import pandas as pd
import urllib.parse
from sqlalchemy import create_engine

print("Starting the database connection process...")

# 1. Read your master file
df_master = pd.read_csv('fleet_telemetry_2024_2026.csv')

# 2. Setup credentials safely
username = 'root'
raw_password = 'Sarthak@123'  # <-- Type your actual password here completely normally

# This safely encodes special characters like @, #, $, or / so SQLAlchemy doesn't break
safe_password = urllib.parse.quote_plus(raw_password)

host = '127.0.0.1'
port = '3306'
database = 'sys' 

# 3. Create the engine with the safe encoded password
engine = create_engine(f"mysql+mysqlconnector://{username}:{safe_password}@{host}:{port}/{database}")

# 4. Push data straight into MySQL
print("Uploading rows to MySQL Workbench... This might take a minute.")
df_master.to_sql('fleet_telemetry', con=engine, if_exists='replace', index=False)

print("Success! Open MySQL Workbench and check the 'fleet_telemetry' table.")

Starting the database connection process...
Uploading rows to MySQL Workbench... This might take a minute.
Success! Open MySQL Workbench and check the 'fleet_telemetry' table.


In [1]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv('fleet_telemetry_2024_2026.csv')

# 2. Europe ke real-world logistics cities ke exact coordinates
euro_hubs = [
    (48.8566, 2.3522),   # Paris (France)
    (45.7640, 4.8357),   # Lyon (France)
    (51.5074, -0.1278),  # London (UK)
    (50.8503, 4.3517),   # Brussels (Belgium)
    (52.3676, 4.9041),   # Amsterdam (Netherlands)
    (43.2965, 5.3698)    # Marseille (France)
]

# 3. Har row ko loop karke ek ek city assign karo taaki data properly spread ho jaye
indices = np.arange(len(df)) % len(euro_hubs)
df['Latitude'] = [euro_hubs[i][0] for i in indices]
df['Longitude'] = [euro_hubs[i][1] for i in indices]

# 4. Save to your shared folder
df.to_csv('fleet_telemetry_2024_2026.csv', index=False)
print("European grid fixed! Clean professional points assigned.")

European grid fixed! Clean professional points assigned.


In [41]:
df_master.shape

(2423, 16)